# Brevitas code
## Define quantized model

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import brevitas.nn as qnn

class VAEnconder(nn.Module):
    def __init__(self):
        super(VAEnconder, self).__init__()
        self.quant_inp = qnn.QuantIdentity()
        self.encoder = nn.Sequential(
            qnn.QuantConv2d(3, 16, 3, stride=2, padding=1, bias=True),  # Increased channels, stride 2
            nn.BatchNorm2d(16),
            nn.ReLU(),
            qnn.QuantConv2d(16, 32, 3, stride=2, padding=1, bias=True),  # Increased channels, stride 2
            nn.BatchNorm2d(32),
            nn.ReLU(),
            qnn.QuantConv2d(32, 64, 3, stride=2, padding=1, bias=True),  # Increased channels, stride 2
            nn.BatchNorm2d(64),
            nn.ReLU(),
            qnn.QuantConv2d(64, 128, 3, stride=2, padding=1, bias=True),  # Increased channels, stride 2
            nn.BatchNorm2d(128),
            nn.ReLU(),
            qnn.QuantConv2d(128, 256, 3, stride=2, padding=1, bias=True),  # Increased channels, stride 2
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.mu = qnn.QuantLinear(256, 6, bias=True)
        self.std = qnn.QuantLinear(256, 6, bias=True)

    def forward(self, x):
        x = self.quant_inp(x)
        a = self.encoder(x).permute(0,2,3,1)
        a = torch.flatten(a, start_dim=1)
        mu = self.mu(a)
        lvar = self.std(a)
        out = torch.cat((mu, lvar), dim=1)
        return out

model = VAEnconder()

## Load pre-trained weights

In [ ]:
import brevitas.config as config
config.IGNORE_MISSING_KEYS = True
model.load_state_dict(torch.load("pre_trained_w_encoder.pt", weights_only=True))

## Finetune pre-trained model

In [ ]:
# WIP
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import datasets, transforms

#-----------------------------------------------------------------------------#
#    IMAGE PREPROCESSING                                                      #
#-----------------------------------------------------------------------------#
## DIRECTORIES with datasets
img_path = 'dataset'

# Dataset with Transformation (using only 100 images)
full_dataset = datasets.ImageFolder(
    img_path,
    transforms.Compose([
        transforms.Resize((128, 256)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.0, 0.0, 0.0], 
            std=[1.0, 1.0, 1.0])
    ])
)

# Data split
trainpctg  = 0.8
train_size = int(trainpctg * len(full_dataset))
test_size = len(full_dataset) - train_size
train_set, val_set = random_split(full_dataset, [train_size, test_size])

print(f"Training set size: {len(train_set)}, Testing set size: {len(val_set)}")

## Test model

In [ ]:
# TO DO

## Export to ONNX

In [ ]:
from brevitas.export import export_qonnx
import torch
import onnx
from finn.util.visualization import showSrc, showInNetron
from finn.util.basic import make_build_dir
import os

build_dir = os.environ["FINN_BUILD_DIR"]
onnx_path = build_dir+"/VAEncoder.onnx"
export_qonnx(model, torch.randn(1, 3, 128, 256), export_path=onnx_path)

# FINN

In [ ]:
from finn.util.test import get_test_model_trained
from qonnx.util.cleanup import cleanup as qonnx_cleanup

qonnx_cleanup(onnx_path, out_file=onnx_path)
showInNetron(onnx_path)

In [ ]:
from qonnx.core.modelwrapper import ModelWrapper
from finn.transformation.qonnx.convert_qonnx_to_finn import ConvertQONNXtoFINN
model = ModelWrapper(onnx_path)
model = model.transform(ConvertQONNXtoFINN())

onnx_path = build_dir+"/VAEncoder_finn.onnx"
model.save(onnx_path)
showInNetron(onnx_path)

In [ ]:
from qonnx.transformation.general import GiveReadableTensorNames, GiveUniqueNodeNames, RemoveStaticGraphInputs
from qonnx.transformation.infer_shapes import InferShapes
from qonnx.transformation.infer_datatypes import InferDataTypes
from qonnx.transformation.fold_constants import FoldConstants

model = model.transform(InferShapes())
model = model.transform(FoldConstants())
model = model.transform(GiveUniqueNodeNames())
model = model.transform(GiveReadableTensorNames())
model = model.transform(InferDataTypes())
model = model.transform(RemoveStaticGraphInputs())

onnx_path = build_dir+"/VAEncoder_tidy.onnx"
model.save(onnx_path)
showInNetron(onnx_path)

## Conversion to HW

In [ ]:
import finn.builder.build_dataflow as build
import finn.builder.build_dataflow_config as build_cfg
import os
import shutil

estimates_output_dir = "output_estimates_only"

#Delete previous run results if exist
if os.path.exists(estimates_output_dir):
    shutil.rmtree(estimates_output_dir)
    print("Previous run results deleted!")


cfg_estimates = build.DataflowBuildConfig(
    output_dir          = estimates_output_dir,
    mvau_wwidth_max     = 80,
    target_fps          = 1000000,
    synth_clk_period_ns = 10.0,
    fpga_part           = "xczu7ev-ffvc1156-2-e",
    steps               = build_cfg.estimate_only_dataflow_steps,
    generate_outputs=[
        build_cfg.DataflowOutputType.ESTIMATE_REPORTS,
    ]
)

In [ ]:
%%time
build.build_dataflow_cfg(onnx_path, cfg_estimates)

In [ ]:
showInNetron("output_estimates_only/intermediate_models/step_create_dataflow_partition.onnx")

In [ ]:
final_output_dir = "output_final"

#Delete previous run results if exist
if os.path.exists(final_output_dir):
    shutil.rmtree(final_output_dir)
    print("Previous run results deleted!")

cfg = build.DataflowBuildConfig(
    output_dir          = final_output_dir,
    mvau_wwidth_max     = 80,
    target_fps          = 1000000,
    synth_clk_period_ns = 5.0,
    board               = "ZCU104",
    shell_flow_type     = build_cfg.ShellFlowType.VIVADO_ZYNQ,
    generate_outputs=[
        build_cfg.DataflowOutputType.BITFILE,
        build_cfg.DataflowOutputType.PYNQ_DRIVER,
        build_cfg.DataflowOutputType.DEPLOYMENT_PACKAGE,
    ]
)

In [ ]:
build.build_dataflow_cfg(onnx_path, cfg)